# Feature selection & encoding

Reads `atp_matches_feature_add.csv`, drops matches that don't reflect a clean
win/loss, selects and cleans the `rel_*`/context columns the model actually
trains on, one-hot encodes categoricals, and builds the direction-agnostic
training target. Writes `atp_matches_feature_engineer.csv` for `model.ipynb`.

## 1. Load feature_add output

In [ ]:
import pandas as pd

df = pd.read_csv("../artifacts/atp_matches_feature_add.csv")
SEED = 42

## 2. Drop non-clean-result matches & the 2017 season

Retirements/walkovers/defaults end a match without it being played out, so they'd
teach the model a misleading relationship between features and outcome; 2017 is
dropped as a full season (e.g. insufficient rolling-stat warm-up history).

In [2]:
# Remove matches that ended in retirement, walkover, or default
df = df[~df["score"].str.contains("RET|Walkover|W/O|DEF", regex=True)].copy()

In [3]:
# ensure tourney_date is datetime (as you showed)
df["tourney_date"] = pd.to_datetime(
    df["tourney_date"],
    errors="coerce"
)
# drop all matches from 2017
df = df[df["tourney_date"].dt.year != 2017].copy()

df.reset_index(drop=True, inplace=True)

## 3. Select training columns & clip height outliers

Narrows `feature_add.ipynb`'s output down to the whitelist the model actually
trains on. `rel_ht` outliers (>50cm difference, physically implausible for two
adult players) get clipped to the median of the valid rows, as a safety net
against any stray bad height data upstream.

In [4]:
keep_cols = [
    # context
    "surface", "tourney_level", "draw_size", "round", "best_of",

    # strength
    "rel_match_elo_pre", "rel_serve_elo_pre", "rel_return_elo_pre", "rel_rank_points",
    "rel_age", "rel_ht",

    # rolling 52-week form
    "rel_ace_rate_52w", "rel_df_rate_52w", "rel_1stIn_pct_52w", "rel_1stWon_pct_52w",
    "rel_2ndWon_pct_52w", "rel_bp_saved_pct_52w", "rel_bp_converted_pct_52w",

    # experience + h2h
    "rel_career_matches", "rel_surface_matches", "rel_h2h_win_diff", "h2h_matches", "rel_return_elo_x_clay",
    "rel_serve_elo_x_grass"
]

# keep only columns that actually exist (safe if you tweak list later)
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

mask = df['rel_ht'].abs() > 50
df.loc[mask, 'rel_ht'] = df.loc[~mask, 'rel_ht'].median()

## 4. Rolling-stat NaN audit

In [5]:
# Defensive fallback: compute_rates() in feature_add.ipynb always returns a real
# number (falling back to tour-average priors when a player has no rolling
# history), so these should never actually be NaN - this is just cheap insurance
# in case that assumption ever changes upstream.
roll_cols = [
    "rel_bp_converted_pct_52w",
    "rel_bp_saved_pct_52w",
    "rel_ace_rate_52w",
    "rel_df_rate_52w",
    "rel_2ndWon_pct_52w",
    "rel_1stWon_pct_52w",
    "rel_1stIn_pct_52w"
]

for c in roll_cols:
    df[c] = df[c].fillna(0.0)


rel_cols = [c for c in df.columns if c.startswith("rel_")]

rel_nan = (
    df[rel_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

rel_nan


rel_rank_points             0.013382
rel_match_elo_pre           0.000000
rel_serve_elo_pre           0.000000
rel_return_elo_pre          0.000000
rel_age                     0.000000
rel_ht                      0.000000
rel_ace_rate_52w            0.000000
rel_df_rate_52w             0.000000
rel_1stIn_pct_52w           0.000000
rel_1stWon_pct_52w          0.000000
rel_2ndWon_pct_52w          0.000000
rel_bp_saved_pct_52w        0.000000
rel_bp_converted_pct_52w    0.000000
rel_career_matches          0.000000
rel_surface_matches         0.000000
rel_h2h_win_diff            0.000000
rel_return_elo_x_clay       0.000000
rel_serve_elo_x_grass       0.000000
dtype: float64

## 5. Encode round & categorical context columns

Rounds are mapped to an ordinal scale (early → late), then `surface`/
`tourney_level`/`round` are one-hot encoded so the model gets fixed-width
numeric input.

In [6]:
round_order = {
    "R128": 1, "R64": 2, "R32": 3, "R16": 4,
    "QF": 5, "SF": 6, "F": 7
}

df["round"] = (
    df["round"]
    .astype("string")
    .str.upper()
    .map(round_order)
    .fillna(0)
    .astype(int)
)

cat_cols = ["surface", "tourney_level", "round"]

for c in cat_cols:
    df[c] = (
        df[c]
        .astype("string")
        .str.upper()
        .fillna("UNK")
    )

# one-hot encode
df = pd.get_dummies(
    df,
    columns=cat_cols,
    prefix=cat_cols,
    prefix_sep="=",
    dummy_na=False
)

## 6. Build the symmetric training target

Every row so far is implicitly "winner vs loser" (`rel_*` = winner − loser), which
would teach the model to just predict "the winner always wins". To make it learn
a direction-agnostic "player A vs player B" relationship instead, a random 50% of
rows get every `rel_*` column negated and `target` flipped to 0 - so the model
sees both "A beat B" and "B beat A" framings of the same underlying feature
relationship.

In [7]:
df['target'] = 1

# 2. Select 50% of the indices to flip
flip_indices = df.sample(frac=0.5, random_state=SEED).index

# 3. Flip the relative features for those rows
# Identify columns that start with 'rel_'
rel_cols = [c for c in df.columns if c.startswith('rel_')]
df.loc[flip_indices, rel_cols] = df.loc[flip_indices, rel_cols] * -1

# 4. Set the target to 0 for those flipped rows
df.loc[flip_indices, 'target'] = 0

## 7. Export

Feeds directly into `model.ipynb` - keep this filename in sync with it.

In [ ]:
df.to_csv('../artifacts/atp_matches_feature_engineer.csv', index=False, header=True)
df